In [ ]:
import yaml
import pandas as pd

In [ ]:
!pip install mygene regex pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.8 MB/s eta 0:00:00


Phase 0 GWAS collection

In [ ]:
import pandas as pd
import requests
#import mygene
import yaml
from pathlib import Path
from time import sleep
import regex as re

# ══════════════════════════════════════════════════════════════════
# USER INPUT — just change these for any disease/paper
# ══════════════════════════════════════════════════════════════════
USER_INPUT = {
    "disease"     : "alzheimer",
    "disease_abbr": "AD",
    "author"      : "Bellenguez_2022_NatGen",
    "year"        : "2022",
    "doi"         : "10.1038/s41588-022-01024-z",
}

P_THRESHOLD  = 5e-8
OUTPUT_DIR   = Path("results/genes")
RAW_DIR      = Path("resources/raw/gwas")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Output filename auto-generated from user input
out_filename = (f"seed_genes"
                f"_{USER_INPUT['disease_abbr']}"
                f"_{USER_INPUT['author']}"
                f"_{USER_INPUT['year']}.tsv")
out_path = OUTPUT_DIR / out_filename

print(f"Disease  : {USER_INPUT['disease']}")
print(f"Author   : {USER_INPUT['author']} {USER_INPUT['year']}")
print(f"DOI      : {USER_INPUT['doi']}")
print(f"Output   : {out_path}")

# ══════════════════════════════════════════════════════════════════
# STEP 1: Find GCST accession from PubMed ID via GWAS Catalog API
# ══════════════════════════════════════════════════════════════════
def find_gcst_from_doi(doi):
    """Search GWAS Catalog for studies matching a DOI."""

    # Step 1: Convert DOI → PubMed ID via NCBI API
    print(f"\nResolving DOI to PubMed ID: {doi}...")
    try:
        ncbi_url = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
        r = requests.get(ncbi_url,
                        params={"ids": doi, "format": "json"},
                        timeout=15)
        records = r.json().get("records", [])
        pubmed_id = records[0].get("pmid") if records else None

        if not pubmed_id:
            # Fallback: try Europe PMC
            epmc_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
            r2 = requests.get(epmc_url,
                             params={"query": f"DOI:{doi}",
                                     "format": "json",
                                     "resultType": "core"},
                             timeout=15)
            results = r2.json().get("resultList", {}).get("result", [])
            pubmed_id = results[0].get("pmid") if results else None

        if not pubmed_id:
            raise RuntimeError(f"Could not resolve DOI to PubMed ID: {doi}")

        print(f"  DOI → PubMed ID: {pubmed_id}")

    except Exception as e:
        raise RuntimeError(f"DOI resolution failed: {e}")

    # Step 2: Use PubMed ID to query GWAS Catalog
    url    = "https://www.ebi.ac.uk/gwas/rest/api/studies/search/findByPublicationIdPubmedId"
    params = {"pubmedId": pubmed_id, "size": 20}

    print(f"  Querying GWAS Catalog for PubMed ID: {pubmed_id}...")
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        studies = r.json().get("_embedded", {}).get("studies", [])
    except Exception as e:
        raise RuntimeError(f"GWAS Catalog API error: {e}")

    results = []
    for s in studies:
        accession = s.get("accessionId", "")
        trait     = s.get("diseaseTrait", {}).get("trait", "")
        sample    = s.get("initialSampleSize", "")
        print(f"  Found: {accession} | trait: {trait} | sample: {sample}")
        results.append({
            "accession" : accession,
            "trait"     : trait,
            "sample"    : sample,
            "pubmed_id" : pubmed_id
        })

    return results


# ══════════════════════════════════════════════════════════════════
# STEP 2: Download association TSV for a GCST accession
# ══════════════════════════════════════════════════════════════════
def download_associations(accession, raw_dir, p_threshold=5e-8):
    """Download GWAS Catalog associations TSV for a given study accession."""
    out_file = raw_dir / f"{accession}_associations.tsv"

    # Return cached file if already downloaded
    if out_file.exists():
        print(f"  Using cached file: {out_file}")
        return out_file

    # Fetch JSON associations from REST API v2
    api_url = f"https://www.ebi.ac.uk/gwas/rest/api/v2/associations?study={accession}"
    print(f"  Fetching associations for {accession} from API...")

    try:
        r = requests.get(api_url, timeout=120)
        r.raise_for_status()
        data = r.json()

        # Extract associations from paginated response (assumes '_embedded' structure)
        associations = data.get('_embedded', {}).get('associations', [])

        if not associations:
            print(f"  No associations found for {accession}")
            return None

        # Convert to DataFrame and filter by p-value threshold
        df = pd.json_normalize(associations)
        if not df.empty:
            # Common p-value fields; adjust based on actual schema (e.g., 'pValue')
            pval_col = df.filter(like='pValue', axis=1).columns[0] if df.filter(like='pValue', axis=1).size > 0 else None
            if pval_col:
                df = df[df[pval_col] <= p_threshold].copy()

        # Save filtered TSV
        df.to_csv(out_file, sep='\t', index=False)
        print(f"  Saved {len(df)} filtered associations → {out_file}")
        return out_file

    except Exception as e:
        print(f"  ⚠ API fetch failed: {e}")
        return None


# ══════════════════════════════════════════════════════════════════
# STEP 3: Parse association file → gene list
# ══════════════════════════════════════════════════════════════════
def clean_gene_list(raw_genes):
    """Fix malformed gene strings like \"['MTAP'\" back to clean symbols."""
    cleaned = []
    for g in raw_genes:
        g = str(g).strip()
        # Remove list artifacts: ['GENE'] or ['GENE1', 'GENE2']
        g = re.sub(r"[\[\]']", "", g)
        # Split on comma in case multiple genes in one string
        parts = [p.strip().upper() for p in g.split(",")]
        for p in parts:
            # Remove any remaining non-gene characters
            p = re.sub(r"[^A-Z0-9\-]", "", p)
            if len(p) >= 2:
                cleaned.append(p)
    return sorted(set(cleaned))

def parse_associations(filepath, accession, p_threshold=5e-8):
    """Extract mapped genes from GWAS Catalog association TSV."""
    df = pd.read_csv(filepath, sep="\t", low_memory=False)
    print(f"  Raw associations: {len(df)}")

    # P-value filter
    pval_col = next((c for c in df.columns
                     if c.upper() in ["P-VALUE", "PVALUE", "P_VALUE"]), None)
    if pval_col:
        df[pval_col] = pd.to_numeric(df[pval_col], errors="coerce")
        df = df[df[pval_col] <= p_threshold]
        print(f"  After p<{p_threshold}: {len(df)} associations")

    # Find gene column
    gene_col = next((c for c in df.columns
                     if "MAPPED_GENE" in c.upper()), None)
    if not gene_col:
        print(f"  ❌ No MAPPED_GENE column. Columns: {df.columns.tolist()}")
        return []

    # ✅ Correct parsing — explode THEN clean
    raw_genes = (df[gene_col]
                 .dropna()
                 .astype(str)
                 .str.replace(" - ", ",", regex=False)   # intergenic A - B → A,B
                 .str.split(r"[,;]+")                    # split into lists
                 .explode()                               # one gene per row
                 .str.strip()
                 .str.upper()
                 .tolist())

    # ✅ Apply clean_gene_list to fix any remaining artifacts
    genes = clean_gene_list(raw_genes)

    # Remove junk symbols
    genes = [g for g in genes if not re.match(
        r'^LOC\d+$|^-$|^$|^NR$|INTERGENIC|^\d+$', g)]
    genes = [g for g in genes if len(g) >= 2]

    print(f"  Raw gene strings : {len(raw_genes)}")
    print(f"  After cleaning   : {len(genes)}")
    print(f"  Sample           : {genes[:10]}")
    return genes


# ══════════════════════════════════════════════════════════════════
# STEP 4: Normalize gene symbols via HGNC REST
# ══════════════════════════════════════════════════════════════════
def normalize_hgnc(gene_list):
    print(f"\nNormalizing {len(gene_list)} genes via HGNC REST...")
    results = {}
    headers = {"Accept": "application/json"}

    for i, gene in enumerate(gene_list):
        if i % 50 == 0:
            print(f"  {i}/{len(gene_list)}")
        try:
            # ✅ fetch/ returns full record including ensembl_gene_id
            r = requests.get(
                f"https://rest.genenames.org/fetch/symbol/{gene}",
                headers=headers, timeout=10)

            docs = r.json().get("response", {}).get("docs", []) if r.ok else []

            # Fallback: try alias if approved symbol not found
            if not docs:
                r2 = requests.get(
                    f"https://rest.genenames.org/fetch/alias_symbol/{gene}",
                    headers=headers, timeout=10)
                docs = r2.json().get("response", {}).get("docs", []) if r2.ok else []

            if not docs:
                continue

            doc = docs[0]

            # uniprot_ids is a list — take first
            uniprot = doc.get("uniprot_ids", [])
            uniprot = uniprot[0] if uniprot else None

            results[gene] = {
                "approved_symbol" : doc.get("symbol", gene),
                "gene_name"       : doc.get("name", ""),
                "ensembl_id"      : doc.get("ensembl_gene_id"),
                "uniprot_id"      : uniprot,
                "hgnc_id"         : doc.get("hgnc_id"),
                "locus_type"      : doc.get("locus_type", ""),
                "location"        : doc.get("location", ""),
                "status"          : doc.get("status", "")
            }
            sleep(0.05)

        except Exception:
            continue

    found  = len(results)
    failed = [g for g in gene_list if g not in results]
    print(f"  ✅ Resolved : {found}/{len(gene_list)}")
    if failed:
        print(f"  ⚠  Failed  : {failed}")
    return results


# ══════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════

# 1. Find GCST accessions
studies = find_gcst_from_doi(USER_INPUT["doi"])

if not studies:
    raise RuntimeError(
        f"No studies found for PubMed ID {USER_INPUT['pubmed_id']}.\n"
        f"Check the PubMed ID or search manually at:\n"
        f"https://www.ebi.ac.uk/gwas/search?query={USER_INPUT['disease']}"
    )


# if not disease_studies:
disease_studies = studies  # fallback to all

print(f"\nUsing study: {disease_studies[0]}")
accession = disease_studies[0]["accession"]

# 3. Download + parse associations
assoc_file = download_associations(accession, RAW_DIR, P_THRESHOLD)
if not assoc_file:
    raise RuntimeError(f"Could not download associations for {accession}")

all_genes = parse_associations(assoc_file, accession, P_THRESHOLD)

if not all_genes:
    raise RuntimeError(
        f"No genes extracted. Check the association file:\n{assoc_file}")

# 4. Normalize via HGNC
hgnc_map = normalize_hgnc(all_genes)

# 5. Build output dataframe
records = []
for gene in all_genes:
    info = hgnc_map.get(gene, {})
    records.append({
        "gene_symbol"    : info.get("approved_symbol", gene),
        "gene_name"      : info.get("gene_name", ""),
        "ensembl_id"     : info.get("ensembl_id"),
        "hgnc_id"        : info.get("hgnc_id"),
        "locus_type"     : info.get("locus_type", ""),
        "gwas_accession" : accession,
        "source"         : f"{USER_INPUT['author']}_{USER_INPUT['year']}",
        "disease"        : USER_INPUT["disease"],
        "mapping_method" : "GWAS_Catalog_API+HGNC_REST"
    })

out_df = (pd.DataFrame(records)
            .drop_duplicates(subset=["gene_symbol"])
            .dropna(subset=["ensembl_id"])
            .sort_values("gene_symbol")
            .reset_index(drop=True))

# Remove pseudogenes and non-standard symbols
bad = out_df["gene_symbol"].str.contains(
    r'\.|^C\d+orf|^CTD-|^AC\d+|^AL\d+|^RP\d+|P$',
    regex=True, na=False)
out_df = out_df[~bad].reset_index(drop=True)

# 6. Save
out_df.to_csv(out_path, sep="\t", index=False, lineterminator="\n")

# Also save as the standard seed file for pipeline
standard_path = OUTPUT_DIR / f"seed_genes_{USER_INPUT['disease_abbr']}.tsv"
out_df.to_csv(standard_path, sep="\t", index=False, lineterminator="\n")

print(f"\n{'='*50}")
print(f" Disease          : {USER_INPUT['disease']}")
print(f" GWAS accession   : {accession}")
print(f" Seed genes       : {len(out_df)}")
print(f" With Ensembl ID  : {out_df['ensembl_id'].notna().sum()}")
print(f" Sample genes     : {out_df['gene_symbol'].head(10).tolist()}")
print(f" Saved → {standard_path}")

Disease  : alzheimer
Author   : Bellenguez_2022_NatGen 2022
DOI      : 10.1038/s41588-022-01024-z
Output   : results/genes/seed_genes_AD_Bellenguez_2022_NatGen_2022.tsv

Resolving DOI to PubMed ID: 10.1038/s41588-022-01024-z...
  DOI → PubMed ID: 35379992
  Querying GWAS Catalog for PubMed ID: 35379992...
  Found: GCST90027158 | trait: Alzheimer's disease | sample: 39,106 European ancestry clinically diagnosed cases, 46,828 European ancestry proxy cases, 401,577 European ancestry controls

Using study: {'accession': 'GCST90027158', 'trait': "Alzheimer's disease", 'sample': '39,106 European ancestry clinically diagnosed cases, 46,828 European ancestry proxy cases, 401,577 European ancestry controls', 'pubmed_id': 35379992}
  Fetching associations for GCST90027158 from API...
  Saved 20 filtered associations → resources/raw/gwas/GCST90027158_associations.tsv
  Raw associations: 20
  After p<5e-08: 2 associations
  Raw gene strings : 3
  After cleaning   : 3
  Sample           : ['LINC031

String PPI preprocessing

In [ ]:
import zipfile
import os

zip_file = "resources.zip"   # name of the zip file
extract_folder = "resources"  # folder where files will be extracted

# Extract the zip file
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("Extraction complete!")

# Optional: check extracted files
print("Extracted files:")
print(os.listdir(extract_folder))

Extraction complete!
Extracted files:
['resources']


In [ ]:
links_file = "resources/resources/resources/raw/string/9606.protein.links.v12.0.txt.gz"

In [ ]:
import pandas as pd

df = pd.read_csv(
    links_file,
    sep=" ",
    compression="gzip"
)
df["combined_score"] = df["combined_score"] / 1000.0

In [ ]:
df = df[df["combined_score"] >= 0.7]

In [ ]:
df["protein1"] = df["protein1"].str.split(".").str[1]
df["protein2"] = df["protein2"].str.split(".").str[1]

In [ ]:
df= df.reset_index(drop=True)

In [ ]:
df

,protein1,protein2,combined_score
0,ENSP00000000233,ENSP00000158762,0.825
1,ENSP00000000233,ENSP00000357048,0.718
2,ENSP00000000233,ENSP00000262305,0.952
3,ENSP00000000233,ENSP00000329419,0.752
4,ENSP00000000233,ENSP00000469035,0.795
...,...,...,...
473855,ENSP00000501277,ENSP00000326630,0.942
473856,ENSP00000501277,ENSP00000263726,0.944
473857,ENSP00000501317,ENSP00000290524,0.780
473858,ENSP00000501317,ENSP00000305071,0.978


In [ ]:
df.to_csv(r"E:\python-ml\bio_project\results\networks\string_bg_raw.tsv",
    sep="\t",
    index=False,
    lineterminator='\n')

mapping proteins from ENSP --> ENSG

In [ ]:
edges= df

In [ ]:
import pandas as pd

info = pd.read_csv(
    "resources/resources/resources/raw/string/9606.protein.info.v12.0.txt.gz",
    sep="\t",
    compression="gzip"
)

info.head()

,#string_protein_id,preferred_name,protein_size,annotation
0,9606.ENSP00000000233,ARF5,180,ADP-ribosylation factor 5; GTP-binding protein...
1,9606.ENSP00000000412,M6PR,277,Cation-dependent mannose-6-phosphate receptor;...
2,9606.ENSP00000001008,FKBP4,459,"Peptidyl-prolyl cis-trans isomerase FKBP4, N-t..."
3,9606.ENSP00000001146,CYP26B1,512,Cytochrome P450 26B1; Involved in the metaboli...
4,9606.ENSP00000002125,NDUFAF7,441,"Protein arginine methyltransferase NDUFAF7, mi..."


In [ ]:
info["protein_id"] = info["#string_protein_id"].str.split(".").str[1]

In [ ]:
info

,#string_protein_id,preferred_name,protein_size,annotation,protein_id
0,9606.ENSP00000000233,ARF5,180,ADP-ribosylation factor 5; GTP-binding protein...,ENSP00000000233
1,9606.ENSP00000000412,M6PR,277,Cation-dependent mannose-6-phosphate receptor;...,ENSP00000000412
2,9606.ENSP00000001008,FKBP4,459,"Peptidyl-prolyl cis-trans isomerase FKBP4, N-t...",ENSP00000001008
3,9606.ENSP00000001146,CYP26B1,512,Cytochrome P450 26B1; Involved in the metaboli...,ENSP00000001146
4,9606.ENSP00000002125,NDUFAF7,441,"Protein arginine methyltransferase NDUFAF7, mi...",ENSP00000002125
...,...,...,...,...,...
19694,9606.ENSP00000501254,ALDH3B2,385,Aldehyde dehydrogenase family 3 member B2; Oxi...,ENSP00000501254
19695,9606.ENSP00000501259,FAM163B,166,Protein FAM163B; Family with sequence similari...,ENSP00000501259
19696,9606.ENSP00000501265,NWD1,1432,NACHT domain- and WD repeat-containing protein...,ENSP00000501265
19697,9606.ENSP00000501277,LDB1,411,LIM domain-binding protein 1; Binds to the LIM...,ENSP00000501277


In [ ]:
[i for i in info['preferred_name'] if 'ENS' in i]

['ENSP00000293826',
 'ENSP00000329467',
 'ENSA',
 'ENSP00000359685',
 'ENSP00000370980',
 'ENSP00000387111',
 'ENSP00000398526',
 'ENSP00000404127',
 'ENSP00000415609',
 'ENSP00000418875',
 'ENSP00000420949',
 'ENSP00000423325',
 'ENSP00000428055',
 'ENSP00000436422',
 'ENSP00000441269',
 'ENSP00000451560',
 'ENSP00000455145',
 'ENSP00000457704',
 'ENSP00000461865',
 'ENSP00000466140',
 'ENSP00000470615',
 'ENSP00000471613',
 'ENSP00000473056',
 'ENSP00000478249',
 'ENSP00000478882',
 'ENSP00000478941',
 'ENSP00000479064',
 'ENSP00000479373',
 'ENSP00000479921',
 'ENSP00000480079',
 'ENSP00000480106',
 'ENSP00000480507',
 'ENSP00000480527',
 'ENSP00000480571',
 'ENSP00000480768',
 'ENSP00000481142',
 'ENSP00000481316',
 'ENSP00000481571',
 'ENSP00000482263',
 'ENSP00000482855',
 'ENSP00000483140',
 'ENSP00000483415',
 'ENSP00000484075',
 'ENSP00000484918',
 'ENSP00000485074',
 'ENSP00000485259',
 'ENSP00000485615',
 'ENSP00000485630',
 'ENSP00000485664',
 'ENSP00000486623',
 'ENSP00000

In [ ]:
protein_to_gene = dict(
    zip(info["protein_id"], info["preferred_name"])
)
protein_to_gene

{'ENSP00000000233': 'ARF5',
 'ENSP00000000412': 'M6PR',
 'ENSP00000001008': 'FKBP4',
 'ENSP00000001146': 'CYP26B1',
 'ENSP00000002125': 'NDUFAF7',
 'ENSP00000002165': 'FUCA2',
 'ENSP00000002596': 'HS3ST1',
 'ENSP00000002829': 'SEMA3F',
 'ENSP00000003084': 'CFTR',
 'ENSP00000003100': 'CYP51A1',
 'ENSP00000003302': 'USP28',
 'ENSP00000004531': 'SLC7A2',
 'ENSP00000004982': 'HSPB6',
 'ENSP00000005178': 'PDK4',
 'ENSP00000005226': 'USH1C',
 'ENSP00000005257': 'RALA',
 'ENSP00000005260': 'BAIAP2L1',
 'ENSP00000005284': 'CACNG3',
 'ENSP00000005286': 'TMEM132A',
 'ENSP00000005340': 'DVL2',
 'ENSP00000005386': 'RPAP3',
 'ENSP00000005587': 'SKAP2',
 'ENSP00000005995': 'PRSS21',
 'ENSP00000006015': 'HOXA11',
 'ENSP00000006053': 'CX3CL1',
 'ENSP00000006275': 'TRAPPC6A',
 'ENSP00000006526': 'WDR54',
 'ENSP00000006658': 'SPATA20',
 'ENSP00000006724': 'CEACAM7',
 'ENSP00000006777': 'RHBDD2',
 'ENSP00000007390': 'TSR3',
 'ENSP00000007414': 'OSBPL7',
 'ENSP00000007699': 'YBX2',
 'ENSP00000007735': 'KR

In [ ]:
protein_to_gene={k:v for k,v in protein_to_gene.items() if 'ENS' not in v}

In [ ]:
len(protein_to_gene)

19558

In [ ]:
edges= df

In [ ]:
edges["geneA"] = edges["protein1"].map(protein_to_gene)
edges["geneB"] = edges["protein2"].map(protein_to_gene)

In [ ]:
edges = edges.dropna(subset=["geneA", "geneB"])

In [ ]:
edges = edges[edges["geneA"] != edges["geneB"]]

In [ ]:
edges

,protein1,protein2,combined_score,geneA,geneB
0,ENSP00000000233,ENSP00000158762,0.825,ARF5,ACAP1
1,ENSP00000000233,ENSP00000357048,0.718,ARF5,COPA
2,ENSP00000000233,ENSP00000262305,0.952,ARF5,RAB11FIP3
3,ENSP00000000233,ENSP00000329419,0.752,ARF5,COPB2
4,ENSP00000000233,ENSP00000469035,0.795,ARF5,COPE
...,...,...,...,...,...
473855,ENSP00000501277,ENSP00000326630,0.942,LDB1,ZFPM1
473856,ENSP00000501277,ENSP00000263726,0.944,LDB1,LHX4
473857,ENSP00000501317,ENSP00000290524,0.780,RFX7,RFX5
473858,ENSP00000501317,ENSP00000305071,0.978,RFX7,RFXANK


In [ ]:
gene_edges = edges[["geneA", "geneB", "combined_score"]].copy()
gene_edges = gene_edges.rename(columns={"combined_score": "weight"})

In [ ]:
gene_edges["sorted_pair"] = gene_edges.apply(
    lambda x: tuple(sorted([x["geneA"], x["geneB"]])), axis=1
)

In [ ]:
gene_edges = gene_edges.drop_duplicates("sorted_pair")
gene_edges = gene_edges.drop(columns="sorted_pair")

In [ ]:
gene_edges=gene_edges.reset_index(drop=True)

In [ ]:
gene_edges['geneA'][178502], gene_edges['geneB'][178502]

('SULT1A4', 'SULT1A1')

In [ ]:
gene_edges.to_csv(r"results\networks\string_bg.tsv", sep="\t")

performed rwr of seed genes using R code of original repo which is inside RWR-M folder

In [ ]:
##Kavya NOTE- please add seed genes back to the final output
##Right now, the final output contains only new candidate genes ranked by RWR, please add seed genes back to it and give it the maximum score as it is a seed gene, So max-score =1 for seed genes
##This step is required to get good modules or good clusters

Performing Diamond on seed genes (Using DIAMOND.py downloaded from original repo)

In [ ]:
###Kavya NOTE- Please add seed genes back in the final output just like you did in RWR
import pandas as pd

# Load your GWAS seeds — adjust path/column to match your file
seeds = pd.read_csv("results/genes/seed_genes_AD.tsv", sep="\t")
seed_genes = seeds["gene_symbol"].dropna().unique()

# DIAMOnD needs a simple one-gene-per-line text file
with open("results/genes/diamond_seeds_AD.txt", "w") as f:
    for g in seed_genes:
        f.write(g + "\n")

print(f"Seed genes written: {len(seed_genes)}")

Seed genes written: 3


In [ ]:
import os
import pandas as pd

# Ensure directory exists
os.makedirs("results/networks", exist_ok=True)

# Save file again properly
gene_edges.to_csv("results/networks/string_bg.tsv", sep="\t", index=False)

# Check file exists
print(os.listdir("results/networks"))

# Now load it
edges = pd.read_csv("results/networks/string_bg.tsv", sep="\t")

# Save edgelist
edges[["geneA", "geneB"]].to_csv(
    "results/networks/string_bg_edgelist.txt",
    sep="\t", index=False, header=False
)

print(f"Edges written: {len(edges)}")

['string_bg.tsv']
Edges written: 236333


In [ ]:
import os
os.makedirs("results/modules", exist_ok=True)

In [ ]:
import subprocess, sys

result = subprocess.run([
    sys.executable, "DIAMOnD.py",
    "results/networks/string_bg_edgelist.txt",  # network
    "results/genes/diamond_seeds_AD.txt",        # seeds
    "200",                                        # genes to add
    "1",                                          # alpha (weight for seeds, default=1)
    "results/modules/diamond_AD_raw.txt"         # output
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

DIAMOnD(): ignoring 6 of 22 seed genes that are not in the network

 results have been saved to 'results/modules/diamond_AD_raw.txt' 


/content/DIAMOnD.py:182: SyntaxWarning: invalid escape sequence '\s'
  p-val = \sum_{n=kb}^{k} HypergemetricPDF(n,k,N,s)
/content/DIAMOnD.py:422: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  p = float(DIAMOnD_node_info[3])



Performing Proximity test on Rwr genes using netcoloc folder containing necessary file(downloaded from original repo)

In [ ]:
import zipfile
import os

zip_file = "netcoloc.zip"   # name of the zip file
extract_folder = "netcoloc"  # folder where files will be extracted

# Extract the zip file
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("Extraction complete!")

# Optional: check extracted files
print("Extracted files:")
print(os.listdir(extract_folder))

Extraction complete!
Extracted files:
['netcoloc']


In [ ]:
!pip install ndex2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.2/78.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 6.6 MB/s eta 0:00:00


In [ ]:
##KAVYA NOTE _ ##You are testing many genes. A raw one-tailed p-value threshold of 0.025 is not multiple-testing corrected. The proximity significance threshold is arbitrary and may need correction for multiple testing
## Prefer FDR correction, for example with Benjamini-Hochberg.
#Then select by adjusted p-value, not raw p-value.

In [ ]:
#Using exact steps to find z_score which they showed in an example_notebook in their github repo.
#After getting z_score we converted into p_val as we will select lower p_val genes as significant genes.


import os
import numpy as np
import pandas as pd
import networkx as nx
from scipy import stats

import sys
import importlib

# Make sure inner folder is a package
os.makedirs("netcoloc/netcoloc", exist_ok=True)
open("netcoloc/netcoloc/__init__.py", "a").close()

# Add the OUTER folder to sys.path
sys.path.insert(0, os.path.abspath("netcoloc"))

# Clear any broken old import
if "netcoloc" in sys.modules:
    del sys.modules["netcoloc"]

# Import explicitly
netprop = importlib.import_module("netcoloc.netprop")
netprop_zscore = importlib.import_module("netcoloc.netprop_zscore")

print("NetColoc loaded successfully")
print(netprop)
print(netprop_zscore)

# ── 1. Load STRING PPI and build LCC ─────────────────────────────────────────
edges = pd.read_csv("results/networks/string_bg.tsv", sep="\t")
G = nx.from_pandas_edgelist(edges, source="geneA", target="geneB")
# largest_cc = max(nx.connected_components(G), key=len)
# G = G.subgraph(largest_cc).copy()
int_nodes = list(G.nodes())
print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ── 2. Precompute individual heats matrix (do this ONCE — takes a few mins) ───

print('\ncalculating w_prime')
w_prime = netprop.get_normalized_adjacency_matrix(G, conserve_heat=True)
print("Computing individual heats matrix (one-time, ~2-5 mins)...")
w_double_prime = netprop.get_individual_heats_matrix(w_prime, alpha=0.5)
print("Done.")

# ── 3. Load seed genes ────────────────────────────────────────────────────────
seeds_df  = pd.read_csv("results/genes/seed_genes_AD.tsv", sep="\t")
seed_genes = list(set(seeds_df["gene_symbol"]) & set(int_nodes))
print(f"Seed genes in network: {len(seed_genes)} / {len(seeds_df)}")

# ── 4. Calculate per-gene proximity z-scores from seed genes ─────────────────

print("Calculating proximity z-scores (1000 permutations)...")
z_scores, Fnew, Fnew_rand = netprop_zscore.calculate_heat_zscores(
    w_double_prime,
    int_nodes,
    dict(G.degree()),
    seed_genes,
    num_reps      = 1000,
    minimum_bin_size = 100
)
print("Done.")

# ── 5. Load RWR genes and filter by proximity z-score ─────────────────────────
rwr_df   = pd.read_csv("expanded_AD_rwr.tsv", sep="\t")
# RWR-M output columns are GeneNames and Score
rwr_genes = set(rwr_df["gene_symbol"]) & set(int_nodes)

# Build results dataframe for RWR genes only
records = []
for gene in rwr_genes:
    if gene in z_scores.index:
        z = z_scores[gene]
        records.append({
            "gene_symbol" : gene,
            "z_score"     : z,
            # one-tailed p-value: P(Z > z) — higher z = more proximal = smaller p
            "p_value"     : 1 - stats.norm.cdf(z),
            "rwr_score"   : rwr_df.loc[rwr_df["gene_symbol"]==gene, "Score"].values[0]
        })

df = pd.DataFrame(records).sort_values("z_score", ascending=False)


# Significant genes
df_sig = df[df["p_value"] < 0.025]

print(f"Total RWR genes scored  : {len(df)}")
print(df.head(10).to_string(index=False))

# Save
df.to_csv("results/genes/proximity_AD_all.tsv", sep="\t", index=False)
df_sig.to_csv("results/genes/proximity_AD_sig.tsv", sep="\t", index=False)

NetColoc loaded successfully
<module 'netcoloc.netprop' from '/content/netcoloc/netcoloc/netprop.py'>
<module 'netcoloc.netprop_zscore' from '/content/netcoloc/netcoloc/netprop_zscore.py'>
Network: 16155 nodes, 236333 edges

calculating w_prime
Computing individual heats matrix (one-time, ~2-5 mins)...
Done.
Seed genes in network: 16 / 22
Calculating proximity z-scores (1000 permutations)...


100%|██████████| 1000/1000 [00:29<00:00, 33.82it/s]


Done.
Total RWR genes scored  : 16137
gene_symbol  z_score      p_value  rwr_score
      DAPK3 8.700498 0.000000e+00   0.013220
     RNASE6 8.519576 0.000000e+00   0.002685
     RNASE1 7.849278 2.109424e-15   0.002683
    PLEKHA5 7.083566 7.024381e-13   0.000159
     PDZD11 7.083566 7.024381e-13   0.013255
     RNASE8 6.716012 9.338308e-12   0.000085
     RNASE4 6.716005 9.338752e-12   0.000085
    FAM155A 6.620318 1.792133e-11   0.001638
     C2CD4A 6.601029 2.041578e-11   0.001566
     RNASE2 6.590714 2.188583e-11   0.002788


Performing leiden clustering on significant genes(Used the exact module author mentioned in their github repo)

In [ ]:
!pip install python-igraph leidenalg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 67.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import networkx as nx

In [ ]:
###Kavya Node-Filter/remove the clusters less then 20 genes.
#iny-cluster filtering is not reflected in the saved df .You filter tiny clusters in cluster_df, but later save df built from all nodes in G. So if you want the saved file to exclude tiny clusters, you should save cluster_df instead or filter df too

In [ ]:
import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg #<----- this is the module made by the author.

# ── Load data ──────────────────────────────────────────────────────────────────
edges = pd.read_csv("results/networks/string_bg.tsv", sep="\t")
rwr_top   = pd.read_csv("results/genes/proximity_AD_sig.tsv", sep="\t")

# RWR score lookup  {gene -> rwr_score}
rwr_lookup = dict(zip(rwr_top["gene_symbol"], rwr_top["rwr_score"]))

# Final node set = top500 RWR genes (seeds already inside RWR results)
selected_nodes = set(rwr_top["gene_symbol"])

# ── Build subgraph with RWR-based edge weights ─────────────────────────────────
G = nx.Graph()

for _, row in edges.iterrows():
    a, b = row["geneA"], row["geneB"]
    if a in selected_nodes and b in selected_nodes:
        rwr_a = rwr_lookup.get(a, 0.0)
        rwr_b = rwr_lookup.get(b, 0.0)
        rwr_edge_weight = (rwr_a + rwr_b) / 2.0     # mean RWR as edge weight
        G.add_edge(a, b, weight=rwr_edge_weight)

print(f"Subgraph nodes : {G.number_of_nodes()}")
print(f"Subgraph edges : {G.number_of_edges()}")

# ── Keep largest connected component ──────────────────────────────────────────
lcc_nodes = max(nx.connected_components(G), key=len)
G = G.subgraph(lcc_nodes).copy()
print(f"LCC nodes : {G.number_of_nodes()}")
print(f"LCC edges : {G.number_of_edges()}")

# ── Convert to iGraph ──────────────────────────────────────────────────────────
node_list       = list(G.nodes())
mapping         = {node: i   for i, node in enumerate(node_list)}
reverse_mapping = {i:   node for node, i in mapping.items()}

edges_with_weight = [
    (mapping[u], mapping[v], data["weight"])
    for u, v, data in G.edges(data=True)
]

ig_graph = ig.Graph(
    n        = len(node_list),
    edges    = [(u, v) for u, v, _ in edges_with_weight],
    directed = False
)
ig_graph.es["weight"] = [w for _, _, w in edges_with_weight]
ig_graph.vs["name"]   = node_list

# ── Leiden clustering ──────────────────────────────────────────────────────────
leiden_partition = leidenalg.find_partition(
    ig_graph,
    leidenalg.RBConfigurationVertexPartition,
    weights          = "weight",
    resolution_parameter = 0.8,
    seed             = 42,
    n_iterations     = 50
)

print(f"\nClusters found : {len(leiden_partition)}")
print(f"Modularity     : {leiden_partition.modularity:.4f}")

# ── Extract results ────────────────────────────────────────────────────────────
leiden_clusters = {
    reverse_mapping[node_id]: comm
    for node_id, comm in enumerate(leiden_partition.membership)
}

cluster_df = pd.DataFrame([
    {
        "gene"      : gene,
        "cluster"   : cluster,
        "rwr_score" : rwr_lookup.get(gene, 0.0),

    }
    for gene, cluster in leiden_clusters.items()
])

# Drop tiny clusters (< 5 genes)
valid_clusters = cluster_df["cluster"].value_counts()
valid_clusters = valid_clusters[valid_clusters >= 5].index
cluster_df     = cluster_df[cluster_df["cluster"].isin(valid_clusters)]

cluster_df = cluster_df.sort_values(["cluster", "rwr_score"], ascending=[True, False])

print(f"\nCluster summary:")
print(cluster_df.groupby("cluster").agg(
    n_genes  = ("gene",      "count"),
    #n_seeds  = ("is_seed",   "sum"),
    mean_rwr = ("rwr_score", "mean"),
    max_rwr  = ("rwr_score", "max")
).to_string())


Subgraph nodes : 743
Subgraph edges : 3289
LCC nodes : 721
LCC edges : 3266

Clusters found : 14
Modularity     : 0.7550

Cluster summary:
         n_genes  mean_rwr   max_rwr
cluster                             
0            149  0.000327  0.001671
1             90  0.000227  0.002909
2             81  0.000426  0.013220
3             57  0.000287  0.001891
4             56  0.000256  0.002823
5             54  0.000244  0.004459
6             51  0.000177  0.000435
7             48  0.000319  0.001275
8             40  0.000450  0.013255
9             32  0.000315  0.002821
10            31  0.000197  0.001323
11            26  0.000036  0.000286


In [ ]:
df = pd.DataFrame({
    "gene_symbol": list(G.nodes()),
    "leiden_cluster": [leiden_clusters[g] for g in G.nodes()]
})

In [ ]:
print(df["leiden_cluster"].value_counts())

leiden_cluster
0     149
1      90
2      81
3      57
4      56
5      54
6      51
7      48
8      40
9      32
10     31
11     26
13      3
12      3
Name: count, dtype: int64


In [ ]:
df.to_csv("results/modules/AD_clusters.tsv", sep="\t", index=False)

print("Clustering complete.")
print("Leiden clusters:", len(set(df["leiden_cluster"])))

Clustering complete.
Leiden clusters: 14


In [ ]:
###KAVYA NOTE- Add enrichment step for the STRING modules

Performing disease-ppi ingestion

In [ ]:
import zipfile
import os

zip_file = "netcoloc.zip"   # name of the zip file
extract_folder = "netcoloc"  # folder where files will be extracted

# Extract the zip file
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("Extraction complete!")

# Optional: check extracted files
print("Extracted files:")
print(os.listdir(extract_folder))

In [ ]:
import zipfile
import os

zip_file = "resources.zip"   # name of the zip file
extract_folder = "resources"  # folder where files will be extracted

# Extract the zip file
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("Extraction complete!")

# Optional: check extracted files
print("Extracted files:")
print(os.listdir(extract_folder))

Extraction complete!
Extracted files:
['resources']


In [ ]:
import zipfile
import pandas as pd
from pathlib import Path
import yaml
import numpy as np

#with open("config/resources.yaml") as f:
    #resources_cfg = yaml.safe_load(f)

# ── Config — change disease_abbr for any disease ───────────────────
DISEASE_ABBR  = "AD"
SEED_FILE     = f"results/genes/seed_genes_{DISEASE_ABBR}.tsv"
INTACT_ZIP    = r"resources/resources/raw/intact/intact.zip"
BIOGRID_ZIP = r"resources/resources/raw/biogrid/BIOGRID-MV-Physical-LATEST.tab3.zip"
OUT_FILE      = f"results/networks/ppi_disease_{DISEASE_ABBR}_weighted.edgelist"

# ── Load seeds ─────────────────────────────────────────────────────
seeds    = pd.read_csv(SEED_FILE, sep="\t")
seed_set = set(seeds["gene_symbol"].str.strip().str.upper())
print(f"Seed genes loaded : {len(seed_set)}")
print(f"Sample seeds      : {sorted(seed_set)[:10]}")

# ══════════════════════════════════════════════════════════════════
# SOURCE 1: IntAct
# ══════════════════════════════════════════════════════════════════
def parse_intact(zip_path, seed_set):
    print(f"\n{'='*50}")
    print("Parsing IntAct...")
    results = []

    with zipfile.ZipFile(zip_path) as z:
        with z.open("intact.txt") as f:
            chunks = pd.read_csv(f, sep="\t", chunksize=20000, low_memory=False)

            for chunk_num, chunk in enumerate(chunks):
                chunk.columns = [c.strip().lstrip("#").strip()
                                  for c in chunk.columns]
                cols_lower = {c.lower(): c for c in chunk.columns}

                def find_col(*kws):
                    for k in kws:
                        for cl, co in cols_lower.items():
                            if k in cl:
                                return co
                    return None

                taxid_a  = find_col("taxid interactor a")
                taxid_b  = find_col("taxid interactor b")
                alias_a  = find_col("alias(es) interactor a")
                alias_b  = find_col("alias(es) interactor b")
                conf_col = find_col("confidence value")

                if not all([taxid_a, taxid_b, alias_a, alias_b]):
                    continue

                # Human-human filter
                human = (
                    chunk[taxid_a].astype(str).str.contains("9606", na=False) &
                    chunk[taxid_b].astype(str).str.contains("9606", na=False)
                )
                chunk = chunk[human].copy()
                if chunk.empty:
                    continue

                # Extract gene symbols
                pat = r'(?:uniprotkb|hgnc):([A-Z][A-Z0-9\-]+)\(gene name\)'
                chunk["geneA"] = (chunk[alias_a].astype(str)
                                  .str.extract(pat, expand=False)
                                  .str.strip().str.upper())
                chunk["geneB"] = (chunk[alias_b].astype(str)
                                  .str.extract(pat, expand=False)
                                  .str.strip().str.upper())

                # MIscore weight
                if conf_col:
                    chunk["weight"] = (chunk[conf_col].astype(str)
                                       .str.extract(r'intact-miscore:([\d.]+)',
                                                    expand=False)
                                       .astype(float))
                else:
                    chunk["weight"] = np.nan

                # Filters
                chunk = chunk.dropna(subset=["geneA", "geneB", "weight"])
                chunk = chunk[chunk["weight"] > 0.4]
                chunk = chunk[chunk["geneA"] != chunk["geneB"]]

                # Seed filter
                seed_mask = (chunk["geneA"].isin(seed_set) |
                             chunk["geneB"].isin(seed_set))
                chunk = chunk[seed_mask].copy()

                if chunk.empty:
                    continue

                results.append(chunk[["geneA", "geneB", "weight"]])
                print(f"  IntAct chunk {chunk_num}: {len(chunk)} edges")

    if not results:
        print("  ❌ No IntAct interactions found")
        return pd.DataFrame(columns=["geneA", "geneB", "weight"])

    df = pd.concat(results, ignore_index=True)
    df["source"] = "IntAct"
    print(f"  ✅ IntAct raw edges: {len(df)}")
    return df


# ══════════════════════════════════════════════════════════════════
# SOURCE 2: BioGRID TAB3
# ══════════════════════════════════════════════════════════════════
def parse_biogrid(zip_path, seed_set):
    print(f"\n{'='*50}")
    print("Parsing BioGRID...")
    results = []

    with zipfile.ZipFile(zip_path) as z:
        tab3_file = next(
            (n for n in z.namelist() if n.endswith(".tab3.txt")), None)

        if not tab3_file:
            print(f"  ❌ No TAB3 file. Files: {z.namelist()}")
            return pd.DataFrame(columns=["geneA", "geneB", "weight"])

        print(f"  Reading: {tab3_file}")

        # ✅ Read full file first without chunking to get real headers
        with z.open(tab3_file) as f:
            # BioGRID TAB3 has a comment line starting with # as header
            # Read first few lines to inspect
            first_lines = []
            for i, line in enumerate(f):
                first_lines.append(line.decode("utf-8"))
                if i >= 3:
                    break

        print("  First 3 lines:")
        for l in first_lines:
            print(f"    {l[:120]}")

        # ✅ Re-open and read properly
        with z.open(tab3_file) as f:
            # Skip comment lines starting with #, use first non-comment as header
            chunks = pd.read_csv(
                f,
                sep       = "\t",
                chunksize = 20000,
                low_memory= False,
                comment   = None,      # don't skip # lines — first line IS the header
                header    = 0,         # first line is header
                encoding  = "utf-8"
            )

            for chunk_num, chunk in enumerate(chunks):

                # Clean column names
                chunk.columns = [c.strip().lstrip("#").strip()
                                  for c in chunk.columns]

                if chunk_num == 0:
                    print(f"  Real columns: {chunk.columns.tolist()}")

                # ✅ BioGRID TAB3 actual column names:
                # "Official Symbol Interactor A"
                # "Official Symbol Interactor B"
                # "Organism Scientific Name Interactor A"
                # "Organism Scientific Name Interactor B"
                # "Experimental System Type"
                # "Score"

                # Find symbol columns flexibly
                sym_a = next((c for c in chunk.columns
                              if "official symbol" in c.lower()
                              and "interactor a" in c.lower()), None)
                sym_b = next((c for c in chunk.columns
                              if "official symbol" in c.lower()
                              and "interactor b" in c.lower()), None)
                org_a = next((c for c in chunk.columns
                              if "organism" in c.lower()
                              and "interactor a" in c.lower()), None)
                org_b = next((c for c in chunk.columns
                              if "organism" in c.lower()
                              and "interactor b" in c.lower()), None)
                score_col = next((c for c in chunk.columns
                                  if "score" in c.lower()), None)

                if not sym_a or not sym_b:
                    if chunk_num == 0:
                        print(f"  ⚠ Symbol columns not found.")
                        print(f"  Available: {chunk.columns.tolist()}")
                    continue

                # Human filter — BioGRID uses organism name not taxid
                if org_a and org_b:
                    human = (
                        chunk[org_a].astype(str).str.contains(
                            "9606|Homo sapiens", na=False, case=False) &
                        chunk[org_b].astype(str).str.contains(
                            "9606|Homo sapiens", na=False, case=False)
                    )
                    chunk = chunk[human].copy()

                if chunk.empty:
                    continue

                chunk["geneA"] = chunk[sym_a].astype(str).str.strip().str.upper()
                chunk["geneB"] = chunk[sym_b].astype(str).str.strip().str.upper()

                # Weight — BioGRID MV-Physical uses score or default 1.0
                if score_col:
                    chunk["weight"] = pd.to_numeric(
                        chunk[score_col], errors="coerce")
                    # Normalize if scores > 1
                    max_s = chunk["weight"].max()
                    chunk["weight"] = (chunk["weight"] / max_s
                                       if max_s > 1 else chunk["weight"])
                    chunk["weight"] = chunk["weight"].fillna(1.0)
                else:
                    chunk["weight"] = 1.0

                # Basic filters
                chunk = chunk[~chunk["geneA"].isin(["-", "nan", ""])]
                chunk = chunk[~chunk["geneB"].isin(["-", "nan", ""])]
                chunk = chunk[chunk["geneA"] != chunk["geneB"]]

                # Seed filter
                seed_mask = (chunk["geneA"].isin(seed_set) |
                             chunk["geneB"].isin(seed_set))
                chunk = chunk[seed_mask].copy()

                if chunk.empty:
                    continue

                results.append(chunk[["geneA", "geneB", "weight"]])
                print(f"  BioGRID chunk {chunk_num}: {len(chunk)} edges")

    if not results:
        print("  ❌ No BioGRID interactions found")
        return pd.DataFrame(columns=["geneA", "geneB", "weight"])

    df = pd.concat(results, ignore_index=True)
    df["source"] = "BioGRID"
    print(f"  ✅ BioGRID raw edges: {len(df)}")
    return df


# ══════════════════════════════════════════════════════════════════
# MERGE IntAct + BioGRID
# ══════════════════════════════════════════════════════════════════
df_intact  = parse_intact(INTACT_ZIP, seed_set)
df_biogrid = parse_biogrid(BIOGRID_ZIP, seed_set)

print(f"\n{'='*50}")
print(f"IntAct edges  : {len(df_intact)}")
print(f"BioGRID edges : {len(df_biogrid)}")

# Combine both sources
df_all = pd.concat([df_intact, df_biogrid], ignore_index=True)

# Normalize to undirected pairs
df_all["pair"] = df_all.apply(
    lambda r: tuple(sorted([r["geneA"], r["geneB"]])), axis=1)

# For duplicate edges across sources — keep max weight
df_merged = (df_all.groupby("pair")
                   .agg(weight=("weight", "max"),
                        sources=("source", lambda x: "+".join(sorted(set(x)))))
                   .reset_index()
                   .assign(
                       geneA=lambda x: x["pair"].apply(lambda p: p[0]),
                       geneB=lambda x: x["pair"].apply(lambda p: p[1])
                   )[["geneA", "geneB", "weight", "sources"]])

# Summary
all_genes     = set(df_merged["geneA"]) | set(df_merged["geneB"])
seeds_covered = seed_set & all_genes
both_sources  = df_merged[df_merged["sources"] == "BioGRID+IntAct"]

print(f"\n✅ Total merged edges     : {len(df_merged)}")
print(f"✅ Unique genes           : {len(all_genes)}")
print(f"✅ Seeds covered          : {len(seeds_covered)}/{len(seed_set)}")
print(f"✅ Edges in BOTH sources  : {len(both_sources)}  ← high confidence")
print(f"✅ Weight range           : {df_merged['weight'].min():.3f} – {df_merged['weight'].max():.3f}")
print(f"✅ Seeds in network       : {sorted(seeds_covered)}")

# Save
out = Path(OUT_FILE)
out.parent.mkdir(parents=True, exist_ok=True)
df_merged.to_csv(out, sep="\t", index=False)
print(f"\n✅ Saved → {out}")

Seed genes loaded : 22
Sample seeds      : ['CCR6', 'CDKN2A-AS1', 'CEP43', 'CLPTM1L', 'CYP19A1', 'EYA2', 'HLA-B', 'HLA-DQA1', 'HLA-DQB1', 'HLA-DRB1']

Parsing IntAct...
  IntAct chunk 1: 13 edges
  IntAct chunk 3: 1 edges
  IntAct chunk 5: 2 edges
  IntAct chunk 7: 8 edges
  IntAct chunk 9: 10 edges
  IntAct chunk 10: 21 edges
  IntAct chunk 12: 3 edges
  IntAct chunk 14: 1 edges
  IntAct chunk 15: 17 edges
  IntAct chunk 16: 1 edges
  IntAct chunk 17: 3 edges
  IntAct chunk 18: 1 edges
  IntAct chunk 19: 7 edges
  IntAct chunk 20: 12 edges
  IntAct chunk 21: 1 edges
  IntAct chunk 23: 5 edges
  IntAct chunk 24: 7 edges
  IntAct chunk 25: 5 edges
  IntAct chunk 26: 18 edges
  IntAct chunk 27: 22 edges
  IntAct chunk 28: 1 edges
  IntAct chunk 29: 4 edges
  IntAct chunk 30: 11 edges
  IntAct chunk 31: 26 edges
  IntAct chunk 32: 4 edges
  IntAct chunk 33: 33 edges
  IntAct chunk 34: 2 edges
  IntAct chunk 35: 11 edges
  IntAct chunk 36: 28 edges
  IntAct chunk 37: 2 edges
  IntAct chunk

In [ ]:
###Kavya NOTE_ repeat the network propagation step (RWR), Proximity based significant genes, clustering similar which is used in STRING based code. So, the code has to be the same, just input files needs to be different.
# This time input is disease-specifc PPI and seed genes.

In [ ]:
#### clustering on disease-ppi. alwways use the LCC for leiden. Remove clusters less then 20 genes
## Perform lieden clustering here similar to the code used for STRING PPI

In [ ]:
!pip install python-igraph leidenalg

In [ ]:
import leidenalg #<---- module made by the author
import igraph as ig
import pandas as pd

# Load IntAct edges
df = pd.read_csv("ppi_disease_AD_weighted.edgelist", sep="\t")

weights = [float(w) for w in df["weight"].tolist()]

# Build weighted graph
g = ig.Graph.TupleList(
    df[["geneA","geneB"]].values,
    weights=df["weight"].astype(float).tolist()
)

# ── Leiden clustering ──────────────────────────────────────────────
leiden_partition = leidenalg.find_partition(
    g, leidenalg.RBConfigurationVertexPartition,
    weights=weights,
    resolution_parameter=1.0,
    n_iterations=50,        # increased from 2 for stability
    seed=42
)



print(f"Leiden  modules : {len(leiden_partition)}")
print(f"Modularity     : {leiden_partition.modularity:.4f}")

# ── Build combined dataframe ───────────────────────────────────────
leiden_map  = {}
louvain_map = {}

for i, cluster in enumerate(leiden_partition):
    for node in cluster:
        leiden_map[g.vs[node]["name"]] = i



# All genes present in graph
all_genes = [g.vs[node]["name"] for node in range(g.vcount())]

modules_df = pd.DataFrame({
    "gene"           : all_genes,
    "leiden_cluster" : [leiden_map.get(gene)  for gene in all_genes],
    "louvain_cluster": [louvain_map.get(gene) for gene in all_genes],
    "leiden_size"    : [len(leiden_partition[leiden_map[gene]])
                        for gene in all_genes],
})

import os
os.makedirs("results/modules", exist_ok=True)

modules_df.to_csv("results/modules/intact_disease_modules_AD.tsv", sep="\t", index=False)

print(f"\nSaved → results/modules/intact_disease_modules_AD.tsv")
print(f"Total genes : {len(modules_df)}")
print(f"\nLeiden cluster sizes:")
print(modules_df["leiden_cluster"].value_counts().sort_index())

Leiden  modules : 15
Modularity     : 0.8531

Saved → results/modules/intact_disease_modules_AD.tsv
Total genes : 211

Leiden cluster sizes:
leiden_cluster
0     35
1     33
2     30
3     24
4     21
5     17
6     12
7     10
8      9
9      4
10     4
11     4
12     3
13     3
14     2
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from itertools import combinations

string_modules = pd.read_csv("results/modules/AD_clusters.tsv",           sep="\t")
intact_modules = pd.read_csv("results/modules/intact_disease_modules_AD.tsv", sep="\t")

string_dict = dict(zip(string_modules["gene_symbol"], string_modules["leiden_cluster"]))
intact_dict = dict(zip(intact_modules["gene"],        intact_modules["leiden_cluster"]))

def jaccard_clustering(labels1, labels2):
    """
    Jaccard similarity between two clusterings.
    Considers all gene pairs and whether they co-cluster.
    """
    n = len(labels1)
    both = 0   # same cluster in BOTH
    either = 0  # same cluster in AT LEAST ONE

    for i, j in combinations(range(n), 2):
        same_1 = labels1[i] == labels1[j]
        same_2 = labels2[i] == labels2[j]

        if same_1 and same_2:
            both += 1
        if same_1 or same_2:
            either += 1

    return both / either if either > 0 else 0.0

common_genes = sorted(set(string_dict) & set(intact_dict))
print(f"\nString genes  : {len(string_dict)}")
print(f"IntAct genes  : {len(intact_dict)}")
print(f"Common genes  : {len(common_genes)}")

string_labels = [string_dict[g] for g in common_genes]
intact_labels = [intact_dict[g] for g in common_genes]

ari = adjusted_rand_score(string_labels, intact_labels)
nmi = normalized_mutual_info_score(string_labels, intact_labels)
jaccard_internal = jaccard_clustering(
    string_labels,
    intact_labels
)

print(f"\nARI : {ari:.4f}   (0=random, 1=identical)")
print(f"NMI : {nmi:.4f}   (0=no mutual info, 1=perfect)")
print(f"\nJACCARD :  {jaccard_internal:.4f}")

# Save comparison
comparison = {
    "string_modules" : len(string_modules["leiden_cluster"].unique()),
    "intact_modules" : len(intact_modules["leiden_cluster"].unique()),
    "common_genes"   : len(common_genes),
    "ari_score"      : round(ari, 4),
    "nmi_score"      : round(nmi, 4),
    "jaccard_similarity": round(jaccard_internal, 4)
}

output_path = "results/comparisons/string_intact_ari_leiden_AD.tsv"

#  IMPORTANT FIX: create directory before saving
Path(output_path).parent.mkdir(parents=True, exist_ok=True)

pd.DataFrame([comparison]).to_csv(output_path, sep="\t", index=False)

print(f"\nSaved → {output_path}")


String genes  : 721
IntAct genes  : 211
Common genes  : 37

ARI : 0.5448   (0=random, 1=identical)
NMI : 0.7435   (0=no mutual info, 1=perfect)

JACCARD :  0.4485

Saved → results/comparisons/string_intact_ari_leiden_AD.tsv


Performing enrichment using the code mam sent

In [ ]:
!pip install gseapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.0/596.0 kB 6.8 MB/s eta 0:00:00


In [ ]:
##Kavya Note-skip plotting/saving when a cluster has no enrichment results
#use only Adjusted P-value for significance claims. Term	Adjusted P-value
#Pathway A	0.0005
#Pathway B	0.03
#Pathway C	0.2 ❌

#Pathway C is not significant, but still shown. Remove pathway C
#avoid overwriting the same CSV in every loop. for eg instead of enrichr_pathway_enrichment_results.csv  (overwritten ❌): write it like this - results/enrichment/module_1/enrichment_full.csv .results/enrichment/module_2/enrichment_full.csv

#MINIMAL working example
#import gseapy as gp
#for i in clusters:

    #gene_list = ...

    #enr = gp.enrichr(...)

    #enrich_df = enr.results

    # 1. Skip empty results
    #if enrich_df is None or enrich_df.empty:
        #print(f"Cluster {i}: No enrichment → skip")
        #continue

    # 2. Use adjusted p-value only
    #enrich_df = enrich_df[enrich_df["Adjusted P-value"] < 0.05]

    #if enrich_df.empty:
        #print(f"Cluster {i}: No significant pathways → skip")
        #continue

    # 3. Save per cluster (no overwrite)
    #outdir = Path(f"results/enrichment/module_{i}")
    #outdir.mkdir(parents=True, exist_ok=True)

    #enrich_df.to_csv(outdir / "enrichment_full.csv", index=False)

In [ ]:
from pathlib import Path
import pandas as pd

gene_module= pd.read_csv("results/modules/intact_disease_modules_AD.tsv", sep="\t")

for i in sorted(gene_module['leiden_cluster'].dropna().unique()): #<----- additionally run the code for all module ids

    # Load gene list
    df_genes = gene_module.loc[gene_module['leiden_cluster'] == i, 'gene']
    gene_list = df_genes.dropna().astype(str).drop_duplicates().tolist()
    print(f"Cluster {i}: {len(gene_list)} genes")

    # Step 3: Run Enrichr enrichment using gseapy
    import gseapy as gp
    print("gseapy installed successfully")

    enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets=['KEGG_2021_Human', 'Reactome_2022'],
    organism='human',
    outdir='enrichr_results',
    cutoff=0.05,
    no_plot=True
    )

    # Step 4: View and save the results
    enrich_df = enr.results
    enrich_df.to_csv("enrichr_pathway_enrichment_results.csv", index=False)
    print("Saved results to 'enrichr_pathway_enrichment_results.csv'")

    # Step 5: Show top 10 KEGG pathways
    top_kegg = enrich_df[enrich_df["Gene_set"] == "KEGG_2021_Human"].sort_values("Adjusted P-value").head(10)
    print("\nTop 10 Enriched KEGG Pathways:")
    display(top_kegg)

    # Show top 10 Reactome pathways
    top_reactome = enrich_df[enrich_df["Gene_set"] == "Reactome_2022"].sort_values("Adjusted P-value").head(10)
    print("\nTop 10 Enriched Reactome Pathways:")
    display(top_reactome)

    # Show top 10 pathways across all gene sets combined
    top_all = enrich_df.sort_values("Adjusted P-value").head(10)
    print("\nTop 10 Enriched Pathways Across All Gene Sets:")
    display(top_all)

    # Step 6: Plot top KEGG pathways
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    top_kegg["-log10(padj)"] = -np.log10(top_kegg["Adjusted P-value"])
    top_kegg["Term"] = top_kegg["Term"].apply(lambda x: x if len(x) <= 60 else x[:57] + "...")

    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_kegg, y="Term", x="-log10(padj)", color="skyblue")
    plt.title("Top 10 Enriched KEGG Pathways")
    plt.xlabel("-log10(Adjusted P-value)")
    plt.ylabel("Pathway")
    plt.tight_layout()
    plt.show()

    # Plot top Reactome pathways
    top_reactome["-log10(padj)"] = -np.log10(top_reactome["Adjusted P-value"])
    top_reactome["Term"] = top_reactome["Term"].apply(lambda x: x if len(x) <= 60 else x[:57] + "...")

    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_reactome, y="Term", x="-log10(padj)", color="lightgreen")
    plt.title("Top 10 Enriched Reactome Pathways")
    plt.xlabel("-log10(Adjusted P-value)")
    plt.ylabel("Pathway")
    plt.tight_layout()
    plt.show()

    # After enrichment and creating top_* dataframes

    # Save CSV files
    Path(f"results/enrichment/module_{i}").mkdir(parents=True, exist_ok=True)
    enrich_df.to_csv(f"results/enrichment/module_{i}/enrichr_pathway_enrichment_results.csv", index=False)
    top_kegg.to_csv(f"results/enrichment/module_{i}/top_10_KEGG_pathways.csv", index=False)
    top_reactome.to_csv(f"results/enrichment/module_{i}/top_10_Reactome_pathways.csv", index=False)
    top_all.to_csv(f"results/enrichment/module_{i}/top_10_all_pathways.csv", index=False)

print("Saved results files.")

Output hidden; open in https://colab.research.google.com to view.